In [1]:
import sys 
!"{sys.executable}" -m pip install networkx numpy scipy torch scikit-learn matplotlib
# torch-geometric may require special install steps on Windows and is optional for this notebook



[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import numpy as np
import networkx as nx
import torch
import torch.nn as nn
import torch.nn.functional as F
import random

# Set random seeds for reproducibility
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)


In [3]:
import os
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt

# Create empty graph
G = nx.Graph()

# Load C_elegans edgelist as an unweighted graph
path = os.path.join("..", "Datasets", "Human12a.edge")

# Each line: V1 V2 weight. We ignore the weight and use binary edges.
edges = np.loadtxt(path, dtype=int, usecols=(0, 1))

# If the file has a single edge, np.loadtxt returns a 1D array, so normalize it.
if edges.ndim == 1:
    edges = edges.reshape(1, 2)

G.add_edges_from(edges)

# Basic info
print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())

# Draw graph only if it is reasonably small; otherwise skip plotting.
if G.number_of_nodes() <= 200:
    pos = nx.spring_layout(G, seed=42)
    nx.draw(G, pos, with_labels=True, node_size=50, font_size=8)
    plt.show()
else:
    print("Graph is large; skipping full plot.")

Nodes: 501
Edges: 6038
Graph is large; skipping full plot.


In [4]:
# adjacency matrix
nodelist = list(G.nodes())
A = nx.to_numpy_array(G, nodelist=nodelist)
print(A)

[[0. 1. 1. ... 0. 0. 0.]
 [1. 0. 1. ... 0. 0. 0.]
 [1. 1. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 1.]
 [0. 0. 0. ... 0. 1. 0.]]


In [5]:
#Degree
deg = np.array([G.degree(n) for n in nodelist])
deg

array([13, 27, 28, 25, 14, 20, 31,  8,  8, 16, 12, 16, 20, 28, 41, 18, 13,
       15, 23, 25, 35, 23, 13, 16, 73, 31, 43, 24, 14, 61, 27, 20, 37, 21,
       24, 36,  8, 12, 19, 19, 56, 34, 20, 34, 55, 35, 32, 40, 37, 26, 26,
       60, 41, 16, 25, 34, 19, 15, 30, 39, 30, 24, 52, 35, 22, 18, 23, 16,
       15,  7,  6, 11,  9, 41, 57, 30, 12,  9, 27, 26, 31, 29, 23, 23, 23,
       17, 31, 45, 39, 16, 25, 20, 26, 30, 25, 15, 16, 11,  6, 15, 12, 18,
       17, 34, 30, 14, 41, 18, 22, 19, 37, 35, 39, 44, 49, 17, 31, 23, 35,
       26, 28, 57, 50, 36, 26, 23, 36, 23, 28, 48, 52, 22, 20, 17, 22, 32,
       19, 22, 33, 22, 16, 40, 22, 39, 35, 24, 30, 19, 17, 23, 14, 45, 26,
       23, 46, 45, 16, 46, 28, 13, 27, 22, 25, 56, 24, 41, 24, 32, 22, 27,
       17, 19, 28, 27, 39, 40, 44, 37, 42, 25, 37, 16, 22, 19, 13, 19, 16,
       17, 20, 46, 20, 21, 14, 22, 20, 24, 18, 21, 17, 24, 32, 26, 41, 34,
       27, 25, 13, 17, 38, 39, 33, 44, 44, 33, 35, 41, 45, 34, 39, 29, 32,
       39, 26, 32, 28, 20

In [6]:
dist = dict(nx.all_pairs_shortest_path_length(G))
dist

{np.int64(1): {np.int64(1): 0,
  np.int64(2): 1,
  np.int64(7): 1,
  np.int64(8): 1,
  np.int64(9): 1,
  np.int64(10): 1,
  np.int64(11): 1,
  np.int64(31): 1,
  np.int64(34): 1,
  np.int64(415): 1,
  np.int64(416): 1,
  np.int64(480): 1,
  np.int64(499): 1,
  np.int64(500): 1,
  np.int64(3): 2,
  np.int64(5): 2,
  np.int64(6): 2,
  np.int64(12): 2,
  np.int64(14): 2,
  np.int64(15): 2,
  np.int64(16): 2,
  np.int64(17): 2,
  np.int64(26): 2,
  np.int64(76): 2,
  np.int64(417): 2,
  np.int64(451): 2,
  np.int64(477): 2,
  np.int64(481): 2,
  np.int64(482): 2,
  np.int64(498): 2,
  np.int64(13): 2,
  np.int64(21): 2,
  np.int64(27): 2,
  np.int64(29): 2,
  np.int64(30): 2,
  np.int64(32): 2,
  np.int64(33): 2,
  np.int64(414): 2,
  np.int64(396): 2,
  np.int64(397): 2,
  np.int64(348): 2,
  np.int64(364): 2,
  np.int64(372): 2,
  np.int64(374): 2,
  np.int64(375): 2,
  np.int64(376): 2,
  np.int64(25): 2,
  np.int64(28): 2,
  np.int64(180): 2,
  np.int64(404): 2,
  np.int64(405): 2,
  n

In [7]:
n = len(nodelist)
dist_matrix = np.zeros((n, n))

for i, u in enumerate(nodelist):
    for j, v in enumerate(nodelist):
        if i != j and v in dist[u]:
            dist_matrix[i, j] = dist[u][v]

print(dist_matrix)

[[0. 1. 1. ... 3. 3. 2.]
 [1. 0. 1. ... 2. 3. 2.]
 [1. 1. 0. ... 2. 3. 2.]
 ...
 [3. 2. 2. ... 0. 3. 2.]
 [3. 3. 3. ... 3. 0. 1.]
 [2. 2. 2. ... 2. 1. 0.]]


4. Node Feature Extraction (Exact Formulas)

Paper defines three measures.

NLI  = Local influence

NGI  = Global influence

NLGC = Hybrid influence

### 4.1 Global Influence (NGI)

**Paper formula:**

$$\text{NGI}_i = \sum_{i \neq j} \frac{\sqrt{d(v_j) + \alpha}}{d_{ij}}$$

**Where:**
*   $d(v_j)$ = degree of node $j$
*   $d_{ij}$ = shortest path distance between node $i$ and node $j$
*   $\alpha$ = constant parameter

### 🔹 Node Global Influence (NGI) – Description

**Definition:**
NGI is a metric used to measure the overall importance of a node by considering its interaction with all other nodes in the network.

**Core Idea:**
It combines both:
*   **Local information** → node degree
*   **Global information** → shortest path distance

**Computation:**
For each node, influence is calculated by summing contributions from all other nodes based on:
*   **Smoothed degree** of the contributing node (using square root scaling)
*   **Distance** between the two nodes

**Role of Degree ($d(v_j)$):**
Nodes with higher connections contribute more influence. However, instead of using the raw degree directly, a square root transformation is applied to moderate the dominance of high-degree nodes.

**Role of Distance ($d_{ij}$):**
Influence decreases as distance increases, ensuring closer nodes have stronger impact. The inverse relationship gives higher weight to nearby nodes.

**Role of $\alpha$ (alpha):**
*   Added inside the square root to stabilize the computation.
*   Prevents zero or very small degree values from reducing influence too much.
*   Helps in smoothing the contribution of nodes.
*   $\alpha = 0.5$ provides a balanced contribution.

**Key Advantage:**
NGI captures both local connectivity and global positioning while ensuring balanced influence using square root scaling.

**Interpretation:**
A node with a higher NGI value is more influential in the network, considering both its connectivity and its position relative to other nodes.

In [8]:
alpha = 0.5

NGI = np.zeros(n)

for i, u in enumerate(nodelist):
    for j, v in enumerate(nodelist):
        if i != j and dist_matrix[i, j] != 0:
            NGI[i] += np.sqrt(deg[j] + alpha) / dist_matrix[i, j]

print("NGI:", NGI)

NGI: [ 755.50726226  913.93493955  927.13768865  875.56483551  833.72105169
  865.1281504   945.60214189  674.09507198  695.83353523  857.07985222
  829.47274056  850.14034918  948.18665638  965.07195228 1036.01435112
  862.84245872  796.74752097  822.75465549  860.85067693  910.83306739
  950.02837413  894.81769716  850.13377906  860.54877588 1153.70477155
  967.44808153 1030.55786164  907.96902593  843.64813049 1180.16848195
 1033.05655527  885.79010699 1072.633316    895.28242101  970.97660995
 1006.84395925  785.23337351  839.41348065  881.58284784  912.41890192
 1167.68779528 1061.8641881  1016.6747656  1067.71469608 1127.81680673
  992.30509542 1038.87079092 1083.80641562 1012.97761283  918.8195951
  980.20309505 1220.44274691 1091.29156303  947.08872355 1044.65391167
 1104.9526575   926.72154363  896.1189355   960.83417801 1098.41539287
 1058.15379066  920.20799584 1179.8111747  1024.94455693  909.70007174
  922.95917853  949.69427801  763.60764773  767.66243417  672.43481784
  

### 4.2 Node Local Influence (NLI)

**Formula:**

$$NLI_i = \frac{d(v_i) \times \log_2 \left( \sum_{j \in N_i} e^{d(v_j)} \right)}{n}$$

---

### 🔹 Node Local Influence (NLI) – Description

**Definition:**
NLI is a metric used to measure the importance of a node based on its local neighborhood structure, focusing only on its immediate connections.

**Core Idea:**
It captures influence using:
*   **Node’s own degree**
*   **Contribution from its neighboring nodes**

**Computation:**
For each node, influence is calculated by:
1. Taking the degree of the node.
2. Multiplying it with the logarithm of the sum of exponential contributions from its neighbors.
3. Normalizing by the total number of nodes ($n$).

**Role of Degree ($d(v_i)$):**
The degree reflects direct connectivity; nodes with more neighbors have higher local influence.

**Role of Neighbor Contribution:**
Each neighbor contributes via an exponential function, which:
*   Amplifies the importance of highly connected neighbors.
*   Highlights strong local structures.

**Role of Logarithm ($\\log$):**
*   Compresses large values from exponential growth.
*   Prevents numerical explosion and ensures balanced scaling.

**Role of Normalization ($n$):**
Dividing by total nodes ensures values are comparable across different graph sizes.

**Key Advantage:**
NLI focuses purely on local structure, making it effective in identifying nodes that are well-connected locally and surrounded by influential neighbors.

**Interpretation:**
A node with higher NLI is more influential within its immediate neighborhood, even if it is not globally central.

In [9]:
NLI = np.zeros(n)

for i, u in enumerate(nodelist):

    neighbors = list(G.neighbors(u))

    # Sum of exponential terms (using neighbor count here as influence proxy)
    exp_sum = 0
    for v in neighbors:
        exp_sum += np.exp(G.degree(v))   # you can modify this part if Ne_i(v_i) defined differently

    if exp_sum > 0:
        NLI[i] = (G.degree(u) * np.log2(exp_sum)) / n
    else:
        NLI[i] = 0

print("NLI:", NLI)

#calculation Verified

NLI: [ 1.16474716  5.67575282  5.88596589  5.25532625  2.9429827   4.204261
  6.5166051   0.57593034  0.57595159  3.3634088   2.5225566   2.81051968
  4.20426135  5.88596589  7.23973473  3.16183464  1.53518848  1.77291128
  3.64273306  5.25532626  6.14893647  4.83490015  2.73276965  2.59458248
 12.89341087  6.51660456  7.71619167  4.21624363  2.45920472 12.82299938
  4.76763715  3.51367798  6.53341915  3.68936188  4.21641358  6.32462037
  0.94457594  1.94593686  3.06392722  3.08106669 11.77193521  7.14724638
  4.20426258  7.14724638 11.5617221   7.35745761  6.72681839  8.40852298
  7.77788352  4.21619665  4.66555446 10.5410551   7.24051226  2.82495293
  4.41529558  6.00498589  2.24425264  1.77175894  5.18333548  8.19831177
  6.30639387  4.14666838  9.35818328  6.0521217   2.59764377  3.16183464
  2.7170787   1.56652486  1.60029408  0.50393892  0.43194754  1.17354899
  1.89191745  8.61873533 11.98214422  6.3063917   1.25482714  0.65121048
  5.67575236  5.46553993  6.51660476  6.09617846

4.3 Hybrid Influence

Paper multiplies them.

$$\text{NLGC}_i = \text{NLI}_i \times \text{NGI}_i$$

In [10]:
NLGC = NLI * NGI
NLGC

array([  879.9749354 ,  5187.26881429,  5457.10081234,  4601.37886079,
        2453.62663022,  3637.22454608,  6162.11573996,   388.23180312,
         400.76643205,  2882.70991546,  2092.39193492,  2389.33618319,
        3986.42451322,  5680.3805942 ,  7500.46907632,  2728.16517537,
        1223.15761552,  1458.67100987,  3135.84922431,  4786.72493386,
        5841.66411961,  4326.35421644,  2323.21978842,  2232.76477483,
       14875.18964166,  6304.47658067,  7951.98198947,  3828.21861938,
        2074.70346538, 15133.29971588,  4925.23881427,  3112.38119677,
        7007.96305019,  3303.02083814,  4094.03896404,  6367.90581406,
         741.71255334,  1633.44562862,  2701.10568064,  2811.22348363,
       13745.94507252,  7589.40497665,  4274.36766974,  7631.21999855,
       13039.50450355,  7300.84267682,  6988.29513835,  9113.21115611,
        7878.82187594,  3873.9240982 ,  4573.19092633, 12864.7542459 ,
        7901.50993599,  2675.48106338,  4612.45579714,  6635.22511782,
      

### 🔹 Multi-Scale Feature Construction

**Definition:**
Multi-scale feature construction is used to capture node influence at different neighborhood levels by progressively aggregating information from neighboring nodes.

**Core Idea:**
Instead of relying on a single-scale measure, influence is computed across multiple levels:
*   **Level 1** → node itself
*   **Level 2** → node + immediate neighbors
*   **Level 3** → node + extended neighborhood

---

### 🧬 Computation: NLI-based Features

**Level 1:**
$$W_{NLI1}(i) = NLI_i$$

**Level 2:**
$$W_{NLI2}(i) = W_{NLI1}(i) + \sum_{j \in N(i)} W_{NLI1}(j)$$

**Level 3:**
$$W_{NLI3}(i) = W_{NLI2}(i) + \sum_{j \in N(i)} W_{NLI2}(j)$$

---

### 🌍 Computation: NGI-based Features

**Level 1:**
$$W_{NGI1}(i) = NGI_i$$

**Level 2:**
$$W_{NGI2}(i) = W_{NGI1}(i) + \sum_{j \in N(i)} W_{NGI1}(j)$$

**Level 3:**
$$W_{NGI3}(i) = W_{NGI2}(i) + \sum_{j \in N(i)} W_{NGI2}(j)$$

---

### ✅ Key Advantages
*   **Higher-Order Influence:** Captures both local and extended neighborhood importance.
*   **Rich Structural Info:** Provides a multidimensional view of a node's position for learning.
*   **Propagation Awareness:** Helps GCNs understand how influence spreads across multiple hops.

**Interpretation:**
Nodes with higher values at deeper levels (e.g., $NLI_3$, $NGI_3$) are not only locally important but are also strategically connected to other influential regions in the graph.

In [11]:
import numpy as np

nodelist = list(G.nodes())
node_index = {node: i for i, node in enumerate(nodelist)}
n = len(nodelist)

# --- NLI Multi-scale ---
W_NLI1 = NLI.copy()
W_NLI2 = np.zeros(n)
W_NLI3 = np.zeros(n)

# NLI2
for i, u in enumerate(nodelist):
    neighbor_sum = 0
    for v in G.neighbors(u):
        j = node_index[v]
        neighbor_sum += W_NLI1[j]
    W_NLI2[i] = W_NLI1[i] + neighbor_sum

# NLI3
for i, u in enumerate(nodelist):
    neighbor_sum = 0
    for v in G.neighbors(u):
        j = node_index[v]
        neighbor_sum += W_NLI2[j]
    W_NLI3[i] = W_NLI2[i] + neighbor_sum


# --- NGI Multi-scale ---
W_NGI1 = NGI.copy()
W_NGI2 = np.zeros(n)
W_NGI3 = np.zeros(n)

# NGI2
for i, u in enumerate(nodelist):
    neighbor_sum = 0
    for v in G.neighbors(u):
        j = node_index[v]
        neighbor_sum += W_NGI1[j]
    W_NGI2[i] = W_NGI1[i] + neighbor_sum

# NGI3
for i, u in enumerate(nodelist):
    neighbor_sum = 0
    for v in G.neighbors(u):
        j = node_index[v]
        neighbor_sum += W_NGI2[j]
    W_NGI3[i] = W_NGI2[i] + neighbor_sum


print("W_NLI1:", W_NLI1)
print("W_NLI2:", W_NLI2)
print("W_NLI3:", W_NLI3)

print("W_NGI1:", W_NGI1)
print("W_NGI2:", W_NGI2)
print("W_NGI3:", W_NGI3)

W_NLI1: [ 1.16474716  5.67575282  5.88596589  5.25532625  2.9429827   4.204261
  6.5166051   0.57593034  0.57595159  3.3634088   2.5225566   2.81051968
  4.20426135  5.88596589  7.23973473  3.16183464  1.53518848  1.77291128
  3.64273306  5.25532626  6.14893647  4.83490015  2.73276965  2.59458248
 12.89341087  6.51660456  7.71619167  4.21624363  2.45920472 12.82299938
  4.76763715  3.51367798  6.53341915  3.68936188  4.21641358  6.32462037
  0.94457594  1.94593686  3.06392722  3.08106669 11.77193521  7.14724638
  4.20426258  7.14724638 11.5617221   7.35745761  6.72681839  8.40852298
  7.77788352  4.21619665  4.66555446 10.5410551   7.24051226  2.82495293
  4.41529558  6.00498589  2.24425264  1.77175894  5.18333548  8.19831177
  6.30639387  4.14666838  9.35818328  6.0521217   2.59764377  3.16183464
  2.7170787   1.56652486  1.60029408  0.50393892  0.43194754  1.17354899
  1.89191745  8.61873533 11.98214422  6.3063917   1.25482714  0.65121048
  5.67575236  5.46553993  6.51660476  6.09617

### 🔹 Neighborhood Matrix Construction

**Definition:**
A neighborhood matrix is constructed for each node to represent its local structural information using a fixed-size subgraph.

**Core Idea:**
Instead of using the entire graph, a localized neighborhood subgraph is extracted for each node, ensuring:
*   Reduced computational complexity
*   Consistent input size for learning models

---

### ⚙️ Computation Steps:
1.  **Extract** one-hop neighbors of the target node.
2.  **Rank** neighbors based on importance scores (e.g., $W_{NLI3}$ or $W_{NGI3}$).
3.  **Select** the top $L$ neighbors.
4.  **Construct** a $(L+1) 	imes (L+1)$ adjacency matrix including the node and selected neighbors.

**Role of Parameter $L$:**
*   Determines the size of the neighborhood.
*   Controls how much local information is captured.
*   Ensures uniform matrix size across all nodes.

---

### ✅ Key Advantage
*   **Efficiency:** Reduces computational complexity.
*   **Robustness:** Avoids bias from high-degree nodes.
*   **Consistency:** Provides structured and consistent input for GCN.

**Interpretation:**
Each node is represented by a fixed-size local subgraph, capturing its most important neighbors and their mutual connections.

In [12]:
import numpy as np

# choose L <= max neighbors: use a fixed neighborhood size of 40 for the large graph
L = 40

def neighborhood_matrix(node):

    nbrs = list(G.neighbors(node))

    # sort neighbors using importance (W_NLI3 or W_NGI3)
    nbrs_sorted = sorted(nbrs, key=lambda x: W_NLI3[nodelist.index(x)], reverse=True)

    nbrs_selected = nbrs_sorted[:L]

    # keep a fixed size L+1; pad with placeholder values if the node has fewer neighbors
    nodes = [node] + nbrs_selected
    if len(nodes) < L + 1:
        nodes += [None] * (L + 1 - len(nodes))

    size = L + 1
    mat = np.zeros((size, size))

    for i, u in enumerate(nodes):
        for j, v in enumerate(nodes):
            if u is not None and v is not None and G.has_edge(u, v):
                mat[i, j] = 1

    return mat, nodes


# Example: for the first node in the graph
mat0, nodes0 = neighborhood_matrix(nodelist[0])

print("Neighborhood Matrix:\n", mat0)
print("Nodes used:", nodes0)

Neighborhood Matrix:
 [[0. 1. 1. ... 0. 0. 0.]
 [1. 0. 1. ... 0. 0. 0.]
 [1. 1. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]
Nodes used: [np.int64(1), np.int64(11), np.int64(500), np.int64(7), np.int64(2), np.int64(499), np.int64(10), np.int64(8), np.int64(480), np.int64(9), np.int64(415), np.int64(416), np.int64(34), np.int64(31), None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None]


### 🔹 Structural Channel Construction

**Definition:**
Structural channel construction embeds node feature information into the neighborhood matrix to generate multiple feature-aware representations of each node.

**Core Idea:**
Instead of using only structural adjacency, node features are incorporated into the matrix to create channels that capture both:
*   **Structural relationships**
*   **Node importance**

---

### ⚙️ Computation:
For each node, a neighborhood matrix is constructed and node features (e.g., NLI, NGI) are embedded into this matrix according to specific rules:

**Channel Construction Rules:**
*   **Diagonal elements:** Represent the feature value of the node itself.
*   **Off-diagonal elements:**
    *   If an edge exists → assign the feature value of the neighbor.
    *   If no edge exists → the value remains zero.

**Channels Created:**
*   **Local influence channels:** $E^{(NLI1)}$, $E^{(NLI2)}$, $E^{(NLI3)}$
*   **Global influence channels:** $E^{(NGI1)}$, $E^{(NGI2)}$, $E^{(NGI3)}$

---

### ✅ Key Advantage
*   **Integration:** Combines structural and feature information seamlessly.
*   **Power:** Enhances the representation power of nodes.
*   **Scalability:** Provides multi-scale learning capability.

**Interpretation:**
Each channel represents a feature-enriched local subgraph, enabling the model to learn both node importance and connectivity patterns simultaneously.

In [13]:
import numpy as np


def embed_channel(mat, nodes, feature_dict):

    size = mat.shape[0]
    out = np.zeros((size, size))

    for i in range(size):
        for j in range(size):

            u = nodes[i]
            v = nodes[j]

            # diagonal → self feature
            if i == j:
                out[i, j] = feature_dict.get(u, 0)

            # edge exists → take neighbor feature
            elif mat[i, j] == 1:
                out[i, j] = feature_dict.get(v, 0)

    return out

### 🔹 Purpose of Structural Channel Construction

Structural channel construction is performed to transform the graph into a format that can effectively capture both **node importance** and **local structural relationships** in a unified representation.

Graph data is inherently irregular, where each node may have a different number of neighbors. This makes it difficult to directly apply deep learning models that require fixed-size inputs.

To address this, a neighborhood matrix is first constructed for each node, ensuring a consistent structure. However, this matrix only represents connectivity and does not include any information about node importance.

Therefore, node features such as local influence (NLI) and global influence (NGI) are embedded into the neighborhood matrix to create **feature-aware channels**.

**In these channels:**
*   The **diagonal elements** represent the importance of the node itself
*   The **off-diagonal elements** represent the importance of neighboring nodes if a connection exists

This transformation allows the model to simultaneously learn:
1.  **Who is connected to whom** (structure)
2.  **How important each node is** (features)

By constructing multiple channels at different scales (NLI1–3 and NGI1–3), the model is able to capture multi-level influence propagation, improving its ability to identify key nodes.

---

### ✅ Key Benefit
This approach enables the graph to be represented as a **multi-channel matrix** (similar to images), making it suitable for deep learning models while preserving both structural and semantic information.

In [14]:
# convert arrays to dict (important)
NLI_dict = {node: NLI[i] for i, node in enumerate(nodelist)}
NGI_dict = {node: NGI[i] for i, node in enumerate(nodelist)}

# example for node 3
mat, nodes = neighborhood_matrix(3)

E_NLI1 = embed_channel(mat, nodes, NLI_dict)
E_NLI2 = embed_channel(mat, nodes, dict(zip(nodelist, W_NLI2)))
E_NLI3 = embed_channel(mat, nodes, dict(zip(nodelist, W_NLI3)))

E_NGI1 = embed_channel(mat, nodes, NGI_dict)
E_NGI2 = embed_channel(mat, nodes, dict(zip(nodelist, W_NGI2)))
E_NGI3 = embed_channel(mat, nodes, dict(zip(nodelist, W_NGI3)))

print("E_NLI1:\n", E_NLI1)
print("E_NLI2:\n", E_NLI2)
print("E_NLI3:\n", E_NLI3)
print("E_NGI1:\n", E_NGI1)
print("E_NGI2:\n", E_NGI2)
print("E_NGI3:\n", E_NGI3)

E_NLI1:
 [[ 7.23973473 11.5617221  12.82299938 ...  1.77291128  1.94593686
   1.53518848]
 [ 7.23973473 11.5617221  12.82299938 ...  0.          1.94593686
   0.        ]
 [ 7.23973473 11.5617221  12.82299938 ...  0.          0.
   0.        ]
 ...
 [ 7.23973473  0.          0.         ...  1.77291128  0.
   1.53518848]
 [ 7.23973473 11.5617221   0.         ...  0.          1.94593686
   0.        ]
 [ 7.23973473  0.          0.         ...  1.77291128  0.
   1.53518848]]
E_NLI2:
 [[231.59265634 362.72017912 371.89378865 ...  76.84810801  63.38194384
   64.10276972]
 [231.59265634 362.72017912 371.89378865 ...   0.          63.38194384
    0.        ]
 [231.59265634 362.72017912 371.89378865 ...   0.           0.
    0.        ]
 ...
 [231.59265634   0.           0.         ...  76.84810801   0.
   64.10276972]
 [231.59265634 362.72017912   0.         ...   0.          63.38194384
    0.        ]
 [231.59265634   0.           0.         ...  76.84810801   0.
   64.10276972]]
E_NLI3:
 [

### 🔹 Channel Tensor Construction

**Definition:**
Channel tensor construction combines multiple feature-embedded neighborhood matrices into a unified multi-dimensional representation for each node.

**Core Idea:**
For each node, six structural channels are generated by embedding different feature representations ($NLI_1$–$NLI_3$ and $NGI_1$–$NGI_3$) into the neighborhood matrix.

---

### ⚙️ Computation:
1.  A **neighborhood matrix** of size $(L+1) \times (L+1)$ is constructed.
2.  **Six feature matrices** are generated by embedding node importance values into the structural layout.
3.  These matrices are stacked to form a tensor of size:
    $$6 \times (L+1) \times (L+1)$$

---

### ✅ Key Advantage
*   **Multi-Perspective:** Captures multiple levels of node importance.
*   **Hybrid Representation:** Combines structural connectivity and feature importance.
*   **Deep Learning Ready:** Enables standard CNN or GCN models to process graph data efficiently.

**Interpretation:**
Each node is represented as a multi-channel tensor, where each channel encodes a different aspect of node influence and neighborhood structure.

In [15]:
channels = []

for node in G.nodes():

    mat, nodes = neighborhood_matrix(node)

    # create feature dicts (node → value), skipping None placeholders
    f1 = {n: NLI_dict.get(n, 0) for n in nodes}
    f2 = {n: W_NLI2[node_index[n]] if n is not None else 0 for n in nodes}
    f3 = {n: W_NLI3[node_index[n]] if n is not None else 0 for n in nodes}

    f4 = {n: NGI_dict.get(n, 0) for n in nodes}
    f5 = {n: W_NGI2[node_index[n]] if n is not None else 0 for n in nodes}
    f6 = {n: W_NGI3[node_index[n]] if n is not None else 0 for n in nodes}

    # create channels
    c1 = embed_channel(mat, nodes, f1)
    c2 = embed_channel(mat, nodes, f2)
    c3 = embed_channel(mat, nodes, f3)

    c4 = embed_channel(mat, nodes, f4)
    c5 = embed_channel(mat, nodes, f5)
    c6 = embed_channel(mat, nodes, f6)

    # stack → (6, L+1, L+1)
    tensor = np.stack([c1, c2, c3, c4, c5, c6])

    channels.append(tensor)

# final shape: (num_nodes, 6, L+1, L+1)
channels = np.array(channels)

print(channels.shape)

(501, 6, 41, 41)


### 🔹 Channel Attention Module

**Definition:**
The channel attention module is used to adaptively learn the importance of different feature channels and enhance the representation of informative channels.

**Core Idea:**
Not all feature channels contribute equally to node importance. Therefore, an attention mechanism is introduced to assign weights to each channel dynamically.

---

### ⚙️ Computation:
1.  **Global average pooling** is applied to each channel to obtain a compact representation.
2.  The pooled values are passed through **fully connected layers**.
3.  A **sigmoid activation** generates normalized weights between 0 and 1.
4.  These weights are **multiplied** with the input feature maps.

---

### ✅ Key Advantage
*   **Feature Selection:** Highlights important feature channels.
*   **Noise Reduction:** Suppresses less relevant information.
*   **Robustness:** Improves model robustness and generalization.

**Interpretation:**
Channels representing more meaningful structural or influence patterns receive higher weights, allowing the model to focus on the most relevant information.

In [16]:
import torch
import torch.nn as nn

class ChannelAttention(nn.Module):

    def __init__(self, channels=6, reduction=2):
        super(ChannelAttention, self).__init__()

        # Global Average Pooling
        self.avg_pool = nn.AdaptiveAvgPool2d(1)

        # Fully Connected Layers (SE block)
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction),
            nn.ReLU(),
            nn.Linear(channels // reduction, channels),
            nn.Sigmoid()
        )

    def forward(self, x):
        # x: (batch, channels, height, width)

        b, c, _, _ = x.size()

        # Step 1: Global Average Pooling
        y = self.avg_pool(x).view(b, c)

        # Step 2: FC → channel weights
        y = self.fc(y).view(b, c, 1, 1)

        # Step 3: Multiply weights
        out = x * y

        return out

In [17]:
# test input
x = torch.randn(2, 6, 3, 3)   # batch=2, channels=6

model = ChannelAttention(6)

out = model(x)

print("Input shape:", x.shape)
print("Output shape:", out.shape)

Input shape: torch.Size([2, 6, 3, 3])
Output shape: torch.Size([2, 6, 3, 3])


### 🔹 NLGCN Model Architecture

**Definition:**
The NLGCN model is a convolutional neural network designed to learn node influence from multi-channel structural representations of graph data.

**Core Idea:**
The model processes the constructed channel tensor using convolutional layers to extract structural patterns, while a channel attention mechanism enhances important feature channels.

---

### 🏗 Architecture:
*   **Input:** Multi-channel tensor of size $6 \times (L+1) \times (L+1)$.
*   **Channel Attention:** Assigns adaptive weights to feature channels.
*   **Convolution Layer 1:** Extracts local structural patterns (followed by Batch Normalization and ReLU).
*   **Pooling Layer:** Reduces spatial dimensions and retains key features.
*   **Convolution Layer 2:** Learns higher-level structural representations.
*   **Fully Connected Layers:** Transform extracted features into the final influence score.

---

### ✅ Key Advantage
*   **Hybrid Learning:** Captures both local and multi-scale structural patterns.
*   **Attention-Driven:** Enhances feature learning using the attention mechanism.
*   **Structured Processing:** Efficiently processes graph data in a consistent matrix format.

**Interpretation:**
The model learns how node importance is influenced by both its local structure and multi-scale neighborhood features, producing a final influence score.

### 🛠 Model Component Summary

| Part | Role |
| :--- | :--- |
| **Channel Attention** | Adaptively select and weight important feature channels |
| **Convolution Layer 1** | Extract local structural patterns from the neighborhood |
| **Max Pooling** | Reduce spatial dimensions and retain significant features |
| **Convolution Layer 2** | Learn higher-order structural representations |
| **Fully Connected** | Map structural features to the final node influence score |

In [18]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class NLGCN(nn.Module):

    def __init__(self):
        super(NLGCN, self).__init__()

        self.attention = ChannelAttention(6)

        # Conv 1
        self.conv1 = nn.Conv2d(6, 16, kernel_size=2)
        self.bn = nn.BatchNorm2d(16)
        self.pool = nn.MaxPool2d(2)

        # -------- Conv 2 (for larger graphs) --------
        # Uncomment this block when using larger graphs (L >= 4 or bigger input size)
        # self.conv2 = nn.Conv2d(16, 32, kernel_size=2)
        # self.pool2 = nn.MaxPool2d(2)

        # -------- FC Layers --------
        # For L=40, input size after conv1+pool is 16 x 20 x 20
        self.fc1 = nn.Linear(16 * 20 * 20, 8)
        self.fc2 = nn.Linear(8, 1)

        # For even larger graphs or extra conv layers, adjust this accordingly
        # self.fc1 = nn.Linear(32 * k * k, 64)  # adjust k based on output size
        # self.fc2 = nn.Linear(64, 1)

    def forward(self, x):

        # x: (batch, 6, L+1, L+1)

        x = self.attention(x)

        x = self.conv1(x)
        x = self.bn(x)
        x = F.relu(x)

        x = self.pool(x)

        # -------- Conv 2 (for larger graphs) --------
        # Uncomment when input size is large enough
        # x = self.conv2(x)
        # x = F.relu(x)
        # x = self.pool2(x)

        x = x.view(x.size(0), -1)

        x = F.relu(self.fc1(x))
        x = self.fc2(x)

        return x

### 🧬 SIR-Based Label Generation

**Definition:**
The SIR (Susceptible–Infected–Recovered) model is used to generate ground truth labels representing node influence.

---

### ⚙️ Computation:

1.  **Epidemic Threshold:** The threshold is calculated as:
    $$\beta_c = \frac{\langle k \rangle}{\langle k^2 \rangle - \langle k \rangle}$$

2.  **Infection Probability:** The probability is set relative to the threshold:
    $$\beta = 1.5\beta_c$$

3.  **Simulation Process:**
    *   Each node is treated as the initial infected node.
    *   The SIR process is simulated multiple times (e.g., 500 runs).
    *   The average number of recovered nodes is calculated as the influence score.

---

### ✅ Normalization:
The labels are normalized to the range $[0, 1]$ to ensure stable model training.

**Interpretation:**
Nodes that infect a larger portion of the network in the SIR simulation are considered more influential and receive higher ground truth scores.

In [19]:
import numpy as np
import random

# ---- Degree calculations ----
deg = np.array([d for n, d in G.degree()])

k_avg = np.mean(deg)
k2_avg = np.mean(deg**2)

beta_c = k_avg / (k2_avg - k_avg)

beta = 1.5 * beta_c
mu = 1


# ---- SIR Simulation ----
def SIR_simulation(G, seed, beta, mu, steps=1000):

    susceptible = set(G.nodes())
    infected = {seed}
    recovered = set()

    susceptible.remove(seed)

    for _ in range(steps):

        new_infected = set()
        new_recovered = set()

        for node in infected:

            # spread infection
            for nbr in G.neighbors(node):
                if nbr in susceptible:
                    if random.random() < beta:
                        new_infected.add(nbr)

            # recovery
            if random.random() < mu:
                new_recovered.add(node)

        infected |= new_infected
        infected -= new_recovered

        recovered |= new_recovered
        susceptible -= new_infected

        if len(infected) == 0:
            break

    return len(recovered)


# ---- Label Generation ----
labels = []
runs = 500

for node in G.nodes():

    spread = 0

    for _ in range(runs):
        spread += SIR_simulation(G, node, beta, mu)

    labels.append(spread / runs)

labels = np.array(labels)


# ---- Normalize Labels ----
labels = labels / np.max(labels)

print("Labels:", labels)

Labels: [0.23168673 0.58567961 0.5775349  0.38854319 0.31917148 0.45201277
 0.64301293 0.06967976 0.0941262  0.3357443  0.2635509  0.37710857
 0.47042153 0.54978622 0.77948914 0.45882672 0.36765775 0.41620563
 0.51323977 0.53661422 0.72929014 0.61535049 0.23067634 0.45339281
 1.         0.64166985 0.8149143  0.54629915 0.37050408 0.96061954
 0.67281935 0.5376246  0.76085858 0.59962788 0.55784467 0.71907537
 0.14840371 0.26324285 0.41799229 0.4348485  0.97349582 0.81087275
 0.53759996 0.7925256  0.95622066 0.80240768 0.75111204 0.7911702
 0.81148884 0.60724275 0.61935508 0.91717289 0.69422231 0.46122947
 0.54021218 0.73935705 0.42964871 0.31971364 0.52233326 0.86232857
 0.78858262 0.48643986 0.96203655 0.57545252 0.40638515 0.43011693
 0.43805217 0.17458753 0.18068682 0.08478628 0.05644615 0.16980667
 0.18783346 0.85780647 0.95678746 0.73851917 0.27901475 0.09120593
 0.56866321 0.65015957 0.77845411 0.66959104 0.54917013 0.45899922
 0.52695393 0.3400446  0.60635558 0.86280912 0.69043952

In [20]:
import torch
import torch.nn as nn

# ---- Convert data to tensors ----
X = torch.tensor(channels, dtype=torch.float32)

# ---- Normalize input channels per channel ----
X = X - X.mean(dim=(0, 2, 3), keepdim=True)
X = X / (X.std(dim=(0, 2, 3), keepdim=True) + 1e-6)

# ---- Normalize labels for stable regression training ----
y = torch.tensor(labels, dtype=torch.float32).view(-1, 1)
y_mean = y.mean()
y_std = y.std()
y = (y - y_mean) / (y_std + 1e-6)

print("X mean per channel:", X.mean(dim=(0, 2, 3)))
print("X std per channel:", X.std(dim=(0, 2, 3)))
print("y mean:", y_mean.item(), "y std:", y_std.item())

# ---- Initialize model ----
model = NLGCN()

# ---- Optimizer ----
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# ---- Loss function ----
criterion = nn.MSELoss()

# ---- Training Loop ----
epochs = 300

for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()
    outputs = model(X)
    loss = criterion(outputs, y)
    loss.backward()
    optimizer.step()
    if epoch % 20 == 0:
        print(f"Epoch {epoch}, Loss = {loss.item():.6f}")

# ---- Final predictions ----
model.eval()
with torch.no_grad():
    predictions = model(X)

print("\nFinal Predictions:\n", predictions)

X mean per channel: tensor([ 7.9358e-08,  2.2452e-07,  1.3864e-07, -8.3344e-09, -2.8322e-07,
        -1.0567e-07])
X std per channel: tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000])
y mean: 0.5036662220954895 y std: 0.20494712889194489
Epoch 0, Loss = 1.420804
Epoch 20, Loss = 0.346323
Epoch 40, Loss = 0.320181
Epoch 60, Loss = 0.300231
Epoch 80, Loss = 0.286593
Epoch 100, Loss = 0.275028
Epoch 120, Loss = 0.264575
Epoch 140, Loss = 0.254853
Epoch 160, Loss = 0.245732
Epoch 180, Loss = 0.237041
Epoch 200, Loss = 0.228757
Epoch 220, Loss = 0.220907
Epoch 240, Loss = 0.213433
Epoch 260, Loss = 0.206341
Epoch 280, Loss = 0.199585

Final Predictions:
 tensor([[-0.5250],
        [ 0.3476],
        [ 0.3329],
        [-0.5172],
        [-0.5250],
        [-0.2335],
        [ 0.6869],
        [-0.5250],
        [-0.5250],
        [-0.5250],
        [-0.5250],
        [-0.5250],
        [-0.2432],
        [ 0.2246],
        [ 1.3269],
        [-0.2490],
        [-0.5250],
        [-0.

### 🔹 Prediction and Ranking Evaluation

**Definition:**
After training, the model predicts influence scores for each node, which are used to rank nodes based on their importance.

---

### ⚙️ Computation:
1.  **Generate Predicted Scores:** The trained model is used to compute influence scores for all nodes in the graph.
2.  **Predicted Ranking:** Nodes are ranked in descending order based on these predicted scores.
3.  **Ground Truth Ranking:** A reference ranking is obtained from the SIR-based simulation labels.
4.  **Comparison:** The predicted ranking is compared with the SIR ranking to measure alignment.

---

### 📊 Evaluation:
*   **Top-k Comparison:** The top-ranked nodes from both predicted and ground truth sets are compared to assess how well the model identifies the most influential nodes.
*   **Ranking Correlation:** Statistical measures can be used to determine the accuracy of the overall node order.

**Key Insight:**
The closer the predicted ranking is to the SIR ranking, the better the model captures the underlying dynamics of node influence within the network.

In [21]:
import numpy as np
import torch

# ---- Prediction ----
model.eval()

with torch.no_grad():
    pred = model(X).detach().cpu().numpy().flatten()

print("Predicted scores:", pred)


# ---- Ranking ----
ranking_pred = np.argsort(pred)[::-1]
ranking_true = np.argsort(labels)[::-1]

print("\nTop predicted nodes:", ranking_pred)
print("Top SIR nodes:", ranking_true)

# Top-k comparison
k = 10
print(f"\nTop {k} predicted nodes:", ranking_pred[:k])
print(f"Top {k} SIR nodes:", ranking_true[:k])

Predicted scores: [-0.5249651   0.34755057  0.3329072  -0.5171932  -0.5249651  -0.23349628
  0.6868637  -0.5249651  -0.5249651  -0.5249651  -0.5249651  -0.5249651
 -0.24324247  0.22455657  1.3268919  -0.24904454 -0.5249651  -0.5249651
  0.08028489  0.21522456  1.0946898   0.5027676  -0.5249651  -0.39814866
  2.385394    0.63587207  1.493371    0.17388976 -0.5229264   2.2199793
  0.7516716   0.2681651   1.2380943   0.4468189   0.25567234  1.0250499
 -0.5249651  -0.5249651  -0.3862374  -0.30384254  2.2581563   1.4723482
  0.18041039  1.3917506   2.2009006   1.4316924   1.2052801   1.3823614
  1.4927359   0.4760124   0.5480524   1.9997668   0.91056544 -0.28045726
  0.22594208  1.1244395  -0.4478544  -0.5249651   0.13675028  1.7160201
  1.3595405  -0.18918559  2.2096226   0.29368097 -0.41926786 -0.37970448
 -0.27826354 -0.5249651  -0.5249651  -0.5249651  -0.5249651  -0.5249651
 -0.5249651   1.7142334   2.1884587   1.1310487  -0.5249651  -0.5249651
  0.43400437  0.70040184  1.3422573   0.78

###  Model Evaluation

**Kendall Tau Correlation:**
The Kendall Tau coefficient is used to measure the similarity between the predicted node ranking and the SIR-based ground truth ranking. A higher value indicates better agreement between the two rankings.

**Top-N Influence Spread:**
The top-N nodes predicted by the model are selected, and their spreading capability is evaluated using the SIR model. The total number of infected nodes represents the effectiveness of the selected nodes.

---

### ✅ Key Insight:
*   **Kendall Tau:** Evaluates the overall ranking consistency.
*   **Top-N Spread:** Evaluates the practical influence performance of the predicted top nodes.

In [22]:
from scipy.stats import kendalltau

# ---- Kendall Tau Correlation ----
tau, p = kendalltau(pred, labels)

print("Kendall Tau:", tau)


# ---- Top-N Influence Spread ----
N = 3   # for small graph (you can change)

top_node_indices = ranking_pred[:N]
top_nodes = [nodelist[idx] for idx in top_node_indices]

spread_total = 0

for node in top_nodes:
    spread_total += SIR_simulation(G, node, beta, mu)

print("Top-N node indices:", top_node_indices)
print("Top-N nodes:", top_nodes)
print("Spread ability:", spread_total)

Kendall Tau: 0.9174508758301053
Top-N node indices: [ 24 419  40]
Top-N nodes: [np.int64(417), np.int64(425), np.int64(348)]
Spread ability: 263


In [23]:
import networkx as nx
from scipy.stats import kendalltau

# Create a copy of G without self-loops for traditional centrality calculations
G_clean = G.copy()
G_clean.remove_edges_from(nx.selfloop_edges(G_clean))

print('Calculating traditional centrality measures for US airports...')

# Degree Centrality
deg_dict = nx.degree_centrality(G_clean)
deg_cent = np.array([deg_dict[n] for n in nodelist])
tau_deg, _ = kendalltau(pred, deg_cent)
print(f'Kendall Tau (Prediction vs Degree): {tau_deg:.4f}')

# Betweenness Centrality
bet_dict = nx.betweenness_centrality(G_clean)
bet_cent = np.array([bet_dict[n] for n in nodelist])
tau_bet, _ = kendalltau(pred, bet_cent)
print(f'Kendall Tau (Prediction vs Betweenness): {tau_bet:.4f}')

# Closeness Centrality
clos_dict = nx.closeness_centrality(G_clean)
clos_cent = np.array([clos_dict[n] for n in nodelist])
tau_clos, _ = kendalltau(pred, clos_cent)
print(f'Kendall Tau (Prediction vs Closeness): {tau_clos:.4f}')

# PageRank
pr_dict = nx.pagerank(G_clean)
pr_cent = np.array([pr_dict[n] for n in nodelist])
tau_pr, _ = kendalltau(pred, pr_cent)
print(f'Kendall Tau (Prediction vs PageRank): {tau_pr:.4f}')

# Coreness (k-core)
core_dict = nx.core_number(G_clean)
core_cent = np.array([core_dict[n] for n in nodelist])
tau_core, _ = kendalltau(pred, core_cent)
print(f'Kendall Tau (Prediction vs Coreness): {tau_core:.4f}')

# Eigenvector Centrality
try:
    eig_dict = nx.eigenvector_centrality(G_clean, max_iter=1000)
    eig_cent = np.array([eig_dict[n] for n in nodelist])
    tau_eig, _ = kendalltau(pred, eig_cent)
    print(f'Kendall Tau (Prediction vs Eigenvector): {tau_eig:.4f}')
except Exception as e:
    print(f'Eigenvector centrality failed: {e}')


Calculating traditional centrality measures for US airports...
Kendall Tau (Prediction vs Degree): 0.7341
Kendall Tau (Prediction vs Betweenness): 0.4246
Kendall Tau (Prediction vs Closeness): 0.5914
Kendall Tau (Prediction vs PageRank): 0.6440
Kendall Tau (Prediction vs Coreness): 0.5964
Kendall Tau (Prediction vs Eigenvector): 0.6945


In [24]:
# ============================================================
# Weighted Centrality Correlation Analysis
# Weight Semantics: HIGH weight = ENEMIES (adversarial/costly)
# So weight is inversely proportional to influence strength.
# We convert: effective_weight = 1 / raw_weight
# This means a high-weight (enemy) edge contributes LESS
# to centrality — reflecting reduced influence flow.
# ============================================================

import numpy as np
import networkx as nx
from scipy.stats import kendalltau

# ---- Step 1: Reload the graph WITH weights ----
# Budapest.txt format: V1  V2  weight (tab/space separated)
path = os.path.join("..", "Datasets",  "Human12a.edge")
raw = np.loadtxt(path)

if raw.ndim == 1:
    raw = raw.reshape(1, -1)

# Build a weighted graph
G_weighted = nx.Graph()
for row in raw:
    u, v, w = int(row[0]), int(row[1]), float(row[2])

    # Safety: avoid zero or negative weights before inversion
    w = abs(w) if w != 0 else 1e-6

    # Inverse weight: high raw weight → low influence (enemy logic)
    inv_w = 1.0 / w

    # If edge already exists keep the minimum inv_w (strongest enemy = weakest link)
    if G_weighted.has_edge(u, v):
        existing = G_weighted[u][v]['weight']
        G_weighted[u][v]['weight'] = min(existing, inv_w)
    else:
        G_weighted.add_edge(u, v, weight=inv_w)

# Remove self-loops (same as unweighted pipeline)
G_weighted.remove_edges_from(nx.selfloop_edges(G_weighted))

print(f"Weighted graph — Nodes: {G_weighted.number_of_nodes()}, Edges: {G_weighted.number_of_edges()}")
print(f"Sample inverse weights: {[round(G_weighted[u][v]['weight'], 4) for u,v in list(G_weighted.edges())[:5]]}")

# ---- Step 2: Compute Weighted Centrality Measures ----
print("\nCalculating weighted centrality measures for Budapest...")

# -- Weighted Degree (Strength) --
# Sum of inv_weights on edges — low-weight enemies reduce strength
strength_dict = dict(G_weighted.degree(weight='weight'))
strength_cent  = np.array([strength_dict.get(n, 0.0) for n in nodelist])
# Normalize to [0,1] for fair comparison
strength_cent  = strength_cent / strength_cent.max() if strength_cent.max() > 0 else strength_cent
tau_wdeg, _ = kendalltau(pred, strength_cent)
print(f"Kendall Tau (Prediction vs Weighted Degree / Strength): {tau_wdeg:.4f}")

# -- Weighted Betweenness --
# Uses inv_weight as distance — high-weight (enemy) edges are LONGER paths
# so they are avoided by shortest paths, reducing betweenness of bridge nodes
wbet_dict  = nx.betweenness_centrality(G_weighted, weight='weight', normalized=True)
wbet_cent  = np.array([wbet_dict[n] for n in nodelist])
tau_wbet, _ = kendalltau(pred, wbet_cent)
print(f"Kendall Tau (Prediction vs Weighted Betweenness):       {tau_wbet:.4f}")

# -- Weighted Closeness --
# distance = inv_weight → enemy edges make nodes "farther apart"
wclos_dict = nx.closeness_centrality(G_weighted, distance='weight')
wclos_cent = np.array([wclos_dict[n] for n in nodelist])
tau_wclos, _ = kendalltau(pred, wclos_cent)
print(f"Kendall Tau (Prediction vs Weighted Closeness):         {tau_wclos:.4f}")

# -- Weighted PageRank --
# edge weight = transition probability proxy (inv_w = low for enemy edges)
# so enemy edges pass less rank to neighbors
wpr_dict   = nx.pagerank(G_weighted, weight='weight')
wpr_cent   = np.array([wpr_dict[n] for n in nodelist])
tau_wpr, _ = kendalltau(pred, wpr_cent)
print(f"Kendall Tau (Prediction vs Weighted PageRank):          {tau_wpr:.4f}")

# -- Weighted Eigenvector Centrality --
# Propagates score proportional to inv_weight of connecting edges
# Enemy edges (high raw w → low inv_w) reduce neighbor's contribution
try:
    weig_dict  = nx.eigenvector_centrality(G_weighted, weight='weight', max_iter=1000)
    weig_cent  = np.array([weig_dict[n] for n in nodelist])
    tau_weig, _ = kendalltau(pred, weig_cent)
    print(f"Kendall Tau (Prediction vs Weighted Eigenvector):       {tau_weig:.4f}")
except nx.PowerIterationFailedConvergence:
    print("Weighted Eigenvector did not converge — trying numpy fallback...")
    weig_dict  = nx.eigenvector_centrality_numpy(G_weighted, weight='weight')
    weig_cent  = np.array([weig_dict[n] for n in nodelist])
    tau_weig, _ = kendalltau(pred, weig_cent)
    print(f"Kendall Tau (Prediction vs Weighted Eigenvector):       {tau_weig:.4f}")

# ---- Step 3: Side-by-side comparison table ----
print("\n" + "="*62)
print(f"{'Measure (Human12a)':<30} {'Unweighted':>12} {'Weighted':>12}")
print("="*62)
print(f"{'Degree / Strength':<30} {tau_deg:>12.4f} {tau_wdeg:>12.4f}")
print(f"{'Betweenness':<30} {tau_bet:>12.4f} {tau_wbet:>12.4f}")
print(f"{'Closeness':<30} {tau_clos:>12.4f} {tau_wclos:>12.4f}")
print(f"{'PageRank':<30} {tau_pr:>12.4f} {tau_wpr:>12.4f}")
print(f"{'Eigenvector':<30} {tau_eig:>12.4f} {tau_weig:>12.4f}")
print("="*62)
print("Weight semantics: high raw weight = adversarial edge")
print("Effective weight = 1 / raw_weight (enemy edges penalized)")

Weighted graph — Nodes: 501, Edges: 6038
Sample inverse weights: [31.4365, 38.7761, 24.1698, 56.3373, 84.6162]

Calculating weighted centrality measures for Budapest...
Kendall Tau (Prediction vs Weighted Degree / Strength): 0.6206
Kendall Tau (Prediction vs Weighted Betweenness):       0.2656
Kendall Tau (Prediction vs Weighted Closeness):         0.1262
Kendall Tau (Prediction vs Weighted PageRank):          0.5464
Kendall Tau (Prediction vs Weighted Eigenvector):       0.4519

Measure (Human12a)               Unweighted     Weighted
Degree / Strength                    0.7341       0.6206
Betweenness                          0.4246       0.2656
Closeness                            0.5914       0.1262
PageRank                             0.6440       0.5464
Eigenvector                          0.6945       0.4519
Weight semantics: high raw weight = adversarial edge
Effective weight = 1 / raw_weight (enemy edges penalized)
